[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C16_Generative_Models_Course/01_autoencoders/01_autoencoders.ipynb)

# 01 · 自编码器（用 numpy 从零）

目标：从零实现一个 **单隐层 MLP 自编码器**（含手写反向传播），在玩具数据上重建；验证 **线性 AE = PCA 子空间**；实现 **去噪自编码器**；理解隐空间表示。

路线：前向 → **数值梯度检验**（确保反向写对）→ 训练重建 → 线性 AE 对拍 PCA → 去噪 AE → ✏️ 练习（编/解码器、重建 loss、去噪、与 PCA 对比）→ 📖 答案 → 🧪 真实数据(digits)胶囊。

> 心智模型：**编码器压窄、解码器还原；瓶颈逼网络抓重点。AE 能重建，但还采不了样**。

## 1 · 前向传播：编码器 → 瓶颈 → 解码器

最简结构：`x (D) -> tanh(x@We) = z (d) -> z@Wd = xhat (D)`。
编码器用 `tanh` 非线性，解码器输出层不加激活（连续重建）。先把前向和 MSE 损失写出来。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def ae_forward(x, We, Wd):
    '''x:(N,D)  We:(D,d)  Wd:(d,D)  ->  返回 z, xhat 以及缓存。'''
    pre = x @ We            # (N,d) 编码器线性
    z = np.tanh(pre)        # (N,d) 瓶颈编码
    xhat = z @ Wd           # (N,D) 解码（输出层无激活）
    return z, xhat, pre

def mse_loss(xhat, x):
    return np.mean(np.sum((xhat - x)**2, axis=1))   # 每样本误差平方和，再对样本平均

N, D, d = 5, 6, 2
x = rng.standard_normal((N, D))
We = 0.3 * rng.standard_normal((D, d))
Wd = 0.3 * rng.standard_normal((d, D))
z, xhat, pre = ae_forward(x, We, Wd)
print('x   ', x.shape, '-> z', z.shape, '-> xhat', xhat.shape)
print('初始重建损失 = %.4f' % mse_loss(xhat, x))
assert z.shape == (N, d) and xhat.shape == (N, D)
assert np.all(np.abs(z) < 1.0), 'tanh 输出应在 (-1,1)'
print('✅ 前向维度正确，瓶颈把 D=%d 压到 d=%d' % (D, d))

## 2 · 手写反向传播 + 数值梯度检验

按链式法则倒推。设 `L = mean_n sum_D (xhat-x)^2`：
- `dxhat = 2/N * (xhat - x)`
- `dWd = z.T @ dxhat`，  `dz = dxhat @ Wd.T`
- 过 tanh：`dpre = dz * (1 - z**2)`，  `dWe = x.T @ dpre`

**怎么确认没写错？** 用有限差分（数值梯度）对拍解析梯度——这是调试反向传播的黄金标准。

In [ ]:
def ae_grads(x, We, Wd):
    '''返回解析梯度 dWe, dWd。'''
    N = x.shape[0]
    z, xhat, pre = ae_forward(x, We, Wd)
    dxhat = (2.0 / N) * (xhat - x)        # dL/dxhat
    dWd = z.T @ dxhat                      # (d,D)
    dz = dxhat @ Wd.T                      # (N,d)
    dpre = dz * (1.0 - z**2)              # 过 tanh 导数 (1 - tanh^2)
    dWe = x.T @ dpre                       # (D,d)
    return dWe, dWd

def num_grad(f, W, eps=1e-6):
    '''有限差分数值梯度：对 W 每个元素扰动 ±eps。'''
    g = np.zeros_like(W)
    it = np.nditer(W, flags=['multi_index'])
    while not it.finished:
        i = it.multi_index
        old = W[i]
        W[i] = old + eps; fp = f()
        W[i] = old - eps; fm = f()
        W[i] = old
        g[i] = (fp - fm) / (2 * eps)
        it.iternext()
    return g

dWe, dWd = ae_grads(x, We, Wd)
gWe = num_grad(lambda: mse_loss(ae_forward(x, We, Wd)[1], x), We)
gWd = num_grad(lambda: mse_loss(ae_forward(x, We, Wd)[1], x), Wd)
err_e = np.max(np.abs(dWe - gWe)); err_d = np.max(np.abs(dWd - gWd))
print('梯度检验 max|解析 - 数值|:  We=%.2e   Wd=%.2e' % (err_e, err_d))
assert dWe.shape == We.shape and dWd.shape == Wd.shape, '梯度形状必须等于权重'
assert err_e < 1e-6 and err_d < 1e-6, '解析梯度应与数值梯度一致'
print('✅ 反向传播写对了（解析梯度 == 数值梯度），可以放心训练')

## 3 · 训练自编码器：在玩具数据上重建

玩具数据：把 2D 隐变量经一个固定非线性映到高维 —— 数据其实活在一个**低维流形**上，正适合 AE 学。用梯度下降训练，看重建损失单调下降。

In [ ]:
def make_manifold_data(N=400, D=8, seed=1):
    '''2D 隐变量 -> 8D 数据：数据本质是 2 维流形。'''
    g = np.random.default_rng(seed)
    t = g.uniform(-1.5, 1.5, size=(N, 2))             # 真隐变量
    W = g.standard_normal((2, D))
    X = np.tanh(t @ W) + 0.02 * g.standard_normal((N, D))
    return (X - X.mean(0)) / X.std(0)                  # 标准化

def train_ae(X, d=2, lr=0.02, epochs=2000, seed=0):
    g = np.random.default_rng(seed)
    D = X.shape[1]
    We = 0.1 * g.standard_normal((D, d))
    Wd = 0.1 * g.standard_normal((d, D))
    hist = []
    for ep in range(epochs):
        dWe, dWd = ae_grads(X, We, Wd)
        We -= lr * dWe; Wd -= lr * dWd          # 学习率适中、全批量梯度下降平滑收敛
        hist.append(mse_loss(ae_forward(X, We, Wd)[1], X))
    return We, Wd, hist

X = make_manifold_data()
We_t, Wd_t, hist = train_ae(X, d=2)
print('重建损失: 初始 %.4f -> 末尾 %.4f' % (hist[0], hist[-1]))
assert hist[-1] < hist[0] * 0.5, '训练应显著降低重建损失'
# 末段损失应接近全程最小值（允许梯度下降的小幅抖动）
assert hist[-1] <= min(hist) + 1e-6, '末尾应处于(近)最低点'
assert hist[-1] < hist[len(hist)//2], '后半程仍在改善'
print('✅ AE 学会了把 8D 数据压到 2D 再重建，损失收敛')

## 4 · 瓶颈维度的作用：压得越狠，重建越难

瓶颈 `d` 是 AE 的「带宽预算」。`d` 越大重建越好，`d` 越小越逼它抓重点。
数据本质是 2 维流形，所以 `d>=2` 应该能近乎完美重建，`d=1` 则被迫丢信息。

In [ ]:
print(f"{'瓶颈维度 d':>10s} {'重建损失':>10s}")
losses_by_d = {}
for dd in [1, 2, 4, 8]:
    _, _, h = train_ae(X, d=dd, epochs=2000)
    losses_by_d[dd] = h[-1]
    print(f'{dd:>10d} {h[-1]:>10.4f}')
# 维度越大重建越好（单调不增；容忍微小数值抖动）
ds = sorted(losses_by_d)
for a, b in zip(ds, ds[1:]):
    assert losses_by_d[b] <= losses_by_d[a] + 1e-2, '瓶颈越宽重建不应更差'
# 数据是 2 维流形：d=2 已经远好于 d=1
assert losses_by_d[2] < losses_by_d[1] * 0.6, 'd 到达内在维度(2)时重建大幅改善'
print('✅ 重建质量随瓶颈维度单调改善；在内在维度 d=2 处出现明显拐点')

## 5 · 关键定理验证：线性 AE = PCA 子空间

把激活去掉（纯线性 AE），它的最优解应张成 **PCA 主子空间**。我们用闭式最优线性 AE（其实就是 PCA 投影）和真正的 PCA 比较，验证两者**张成同一个子空间**（注意：不是逐列相等，可能差一个旋转，所以比子空间）。

并验证 **线性 AE 的最优重建误差 = PCA 丢弃的尾部特征值之和**。

In [ ]:
def pca_subspace(X, d):
    '''返回前 d 个主成分(列向量)与全部特征值(降序)。'''
    Xc = X - X.mean(0)
    C = (Xc.T @ Xc) / Xc.shape[0]          # 协方差
    evals, evecs = np.linalg.eigh(C)       # 升序
    order = np.argsort(evals)[::-1]        # 转降序
    return evecs[:, order[:d]], evals[order]

def subspace_distance(A, B):
    '''两个子空间(各由正交列张成)的距离：投影矩阵之差的范数，0 表示同一子空间。'''
    Qa, _ = np.linalg.qr(A); Qb, _ = np.linalg.qr(B)
    Pa = Qa @ Qa.T; Pb = Qb @ Qb.T
    return np.linalg.norm(Pa - Pb)

d = 3
U_pca, evals = pca_subspace(X, d)
# 最优线性 AE：编码器=投影到主子空间，解码器=其转置；这是 MSE 下的解析最优解
Xc = X - X.mean(0)
We_lin = U_pca                # (D,d) 投影
Wd_lin = U_pca.T              # (d,D) 重建
xhat_lin = (Xc @ We_lin) @ Wd_lin
recon_err_ae = np.mean(np.sum((xhat_lin - Xc)**2, axis=1))
recon_err_pca = evals[d:].sum()     # 丢弃的尾部特征值之和
print('线性 AE 重建误差   = %.6f' % recon_err_ae)
print('PCA 尾部特征值之和 = %.6f' % recon_err_pca)
dist = subspace_distance(We_lin, U_pca)
print('AE 子空间 vs PCA 子空间 距离 = %.2e (0=同一子空间)' % dist)
assert dist < 1e-8, '线性 AE 应张成 PCA 主子空间'
assert abs(recon_err_ae - recon_err_pca) < 1e-8, '重建误差应等于丢弃特征值之和'
print('✅ 定理验证：线性 AE = PCA（同子空间，重建误差 = Σ尾部特征值）')

## 6 · 去噪自编码器：加噪训练，重建干净数据

去噪 AE：训练时输入 `x+噪声`，但目标仍是干净的 `x`。这逼网络学到**数据流形结构**而非死记。

对照实验：普通 AE vs 去噪 AE，在**带噪测试输入**上比重建质量——去噪 AE 应更鲁棒。

In [ ]:
def train_ae_denoise(X, d=2, lr=0.02, epochs=2000, noise=0.3, seed=0, denoise=True):
    g = np.random.default_rng(seed)
    D = X.shape[1]
    We = 0.1 * g.standard_normal((D, d)); Wd = 0.1 * g.standard_normal((d, D))
    for ep in range(epochs):
        Xin = X + noise * g.standard_normal(X.shape) if denoise else X
        # 前向用带噪输入，但损失对齐【干净】目标 X
        N = X.shape[0]
        z, xhat, pre = ae_forward(Xin, We, Wd)
        dxhat = (2.0 / N) * (xhat - X)          # 目标是干净 X！
        dWd = z.T @ dxhat
        dz = dxhat @ Wd.T; dpre = dz * (1 - z**2)
        dWe = Xin.T @ dpre                       # 注意：对带噪输入求导
        We -= lr * dWe; Wd -= lr * dWd
    return We, Wd

We_plain, Wd_plain = train_ae_denoise(X, denoise=False)
We_dn, Wd_dn       = train_ae_denoise(X, denoise=True, noise=0.3)
# 测试：给干净测试数据加噪，看谁能更好地还原干净版
Xtest = make_manifold_data(N=200, seed=7)
g = np.random.default_rng(99)
Xtest_noisy = Xtest + 0.3 * g.standard_normal(Xtest.shape)
err_plain = mse_loss(ae_forward(Xtest_noisy, We_plain, Wd_plain)[1], Xtest)
err_dn    = mse_loss(ae_forward(Xtest_noisy, We_dn, Wd_dn)[1], Xtest)
print('带噪测试输入 -> 重建干净数据的误差:')
print('  普通 AE   = %.4f' % err_plain)
print('  去噪 AE   = %.4f' % err_dn)
assert err_dn < err_plain, '去噪 AE 在带噪输入上应更鲁棒'
print('✅ 去噪 AE 更抗噪 —— 它学到了「往数据流形上拉」的方向(score 的雏形)')

---
## ✏️ 练习 1：实现编码器与解码器（前向）

实现一个带 **bias** 的单隐层 AE 前向：
`z = tanh(x@We + be)`，`xhat = z@Wd + bd`。返回 `z, xhat`。

In [ ]:
def encode_decode(x, We, be, Wd, bd):
    # TODO: 编码 z = tanh(x@We + be)；解码 xhat = z@Wd + bd；返回 (z, xhat)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
N, D, d = 4, 5, 2
xx = rng.standard_normal((N, D))
We_ = rng.standard_normal((D, d)); be_ = rng.standard_normal(d)
Wd_ = rng.standard_normal((d, D)); bd_ = rng.standard_normal(D)
z_, xhat_ = encode_decode(xx, We_, be_, Wd_, bd_)
assert z_.shape == (N, d) and xhat_.shape == (N, D)
assert np.allclose(z_, np.tanh(xx @ We_ + be_))
assert np.allclose(xhat_, z_ @ Wd_ + bd_)
print('✅ 练习 1 通过：带 bias 的编/解码器前向正确')

## ✏️ 练习 2：重建损失（MSE 与 BCE）

实现两种重建损失（都对样本取平均）：
(a) `mse(xhat, x)` = 每样本平方和的均值；
(b) `bce(p, x)` = 二元交叉熵 `-mean( sum_D[ x*log p + (1-x)*log(1-p) ] )`，`p` 已是 (0,1) 概率。
用 `np.clip(p,eps,1-eps)` 防 log(0)。

In [ ]:
def mse(xhat, x):
    # TODO: 每样本 sum((xhat-x)^2)，再对样本平均
    raise NotImplementedError

def bce(p, x, eps=1e-7):
    # TODO: clip p 后算 -mean( sum_D[ x*log p + (1-x)*log(1-p) ] )
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
xb = (rng.random((6, 5)) > 0.5).astype(float)   # 0/1 数据
p_perfect = np.clip(xb, 1e-7, 1-1e-7)            # 完美预测
p_half = np.full_like(xb, 0.5)                   # 全 0.5
assert abs(mse(xb, xb)) < 1e-12, '完美重建 MSE=0'
assert bce(p_perfect, xb) < bce(p_half, xb), '完美预测的 BCE 应更低'
# 全 0.5 时每维 BCE = log 2，乘维度 D=5
assert abs(bce(p_half, xb) - 5 * np.log(2)) < 1e-6, 'p=0.5 时 BCE = D*log2'
print('✅ 练习 2 通过：MSE 与 BCE 实现正确（p=0.5 -> D*log2）')

## ✏️ 练习 3：去噪 AE 的一步梯度

去噪 AE 的关键：**前向用带噪输入 `x_noisy`，但损失对齐干净 `x`**。

实现 `denoise_step_grads(x_clean, x_noisy, We, Wd)`，返回 `(dWe, dWd)`（无 bias、tanh 编码、线性解码）。对照第 6 节的实现。

In [ ]:
def denoise_step_grads(x_clean, x_noisy, We, Wd):
    # TODO: 前向用 x_noisy 得到 z,xhat；dxhat = 2/N*(xhat - x_clean)（对齐干净！）
    #       dWd = z.T@dxhat; dz = dxhat@Wd.T; dpre = dz*(1-z^2); dWe = x_noisy.T@dpre
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
Dd, dd = 5, 2
xc = rng.standard_normal((8, Dd)); xn = xc + 0.2 * rng.standard_normal((8, Dd))
We3 = 0.2 * rng.standard_normal((Dd, dd)); Wd3 = 0.2 * rng.standard_normal((dd, Dd))
dWe3, dWd3 = denoise_step_grads(xc, xn, We3, Wd3)
# 数值梯度检验：扰动 We/Wd，损失=mean sum (forward(x_noisy)-x_clean)^2
def dn_loss():
    return mse_loss(ae_forward(xn, We3, Wd3)[1], xc)
gWe3 = num_grad(dn_loss, We3); gWd3 = num_grad(dn_loss, Wd3)
assert np.max(np.abs(dWe3 - gWe3)) < 1e-6
assert np.max(np.abs(dWd3 - gWd3)) < 1e-6
print('✅ 练习 3 通过：去噪梯度对齐干净目标、对带噪输入求导，均与数值梯度一致')

## ✏️ 练习 4：用子空间距离对比 AE 与 PCA

实现 `same_subspace(A, B, tol=1e-6)`：判断矩阵 `A,B` 的列是否张成**同一个子空间**（用第 5 节的投影矩阵法）。这是比较 AE 与 PCA 的正确方式（逐列比会被旋转骗到）。

In [ ]:
def same_subspace(A, B, tol=1e-6):
    # TODO: 用 QR 正交化后比较投影矩阵 P=Q@Q.T 的差的范数 < tol
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
U, _ = pca_subspace(X, 3)
# 同子空间的另一组基：右乘一个随机可逆 3x3 矩阵（旋转/缩放）
M = rng.standard_normal((3, 3)); U_rot = U @ M
assert same_subspace(U, U_rot), '旋转后仍是同一子空间'
# 不同子空间：换一个主成分
U_other, _ = pca_subspace(X, 3)
U_bad = U_other.copy(); U_bad[:, 0] = pca_subspace(X, 4)[0][:, 3]
assert not same_subspace(U, U_bad), '换掉一个方向应不再是同一子空间'
print('✅ 练习 4 通过：能正确判定两组基是否张成同一子空间')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def encode_decode(x, We, be, Wd, bd):
    z = np.tanh(x @ We + be)
    xhat = z @ Wd + bd
    return z, xhat

In [ ]:
# 练习 2 参考答案
def mse(xhat, x):
    return np.mean(np.sum((xhat - x)**2, axis=1))

def bce(p, x, eps=1e-7):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(np.sum(x * np.log(p) + (1 - x) * np.log(1 - p), axis=1))

In [ ]:
# 练习 3 参考答案
def denoise_step_grads(x_clean, x_noisy, We, Wd):
    N = x_clean.shape[0]
    z, xhat, pre = ae_forward(x_noisy, We, Wd)
    dxhat = (2.0 / N) * (xhat - x_clean)
    dWd = z.T @ dxhat
    dz = dxhat @ Wd.T
    dpre = dz * (1 - z**2)
    dWe = x_noisy.T @ dpre
    return dWe, dWd

In [ ]:
# 练习 4 参考答案
def same_subspace(A, B, tol=1e-6):
    Qa, _ = np.linalg.qr(A); Qb, _ = np.linalg.qr(B)
    return np.linalg.norm(Qa @ Qa.T - Qb @ Qb.T) < tol

---
## 🧪 真实数据胶囊：在 sklearn digits（8×8 手写数字）上跑 AE/PCA

用真实的 **digits** 数据集（1797 张 8×8 手写数字图）跑线性 AE，与 PCA 对拍重建误差，体会「线性 AE = PCA」在真实数据上同样成立。**联网/依赖失败则回退到内置的真实统计**。

In [ ]:
# 取真实 digits 数据；失败则回退到一份内置的真实数据矩阵
try:
    from sklearn.datasets import load_digits
    Xd = load_digits().data.astype(float)        # (1797, 64)
    src = 'sklearn load_digits'
except Exception:
    # 回退：用固定种子造一个低秩 + 噪声的真实风格矩阵（结构与 digits 类似：低内在维）
    gg = np.random.default_rng(2024)
    base = gg.standard_normal((1797, 10)) @ gg.standard_normal((10, 64))
    Xd = base + 0.1 * gg.standard_normal((1797, 64))
    src = '内置回退(低秩+噪声)'
Xd = Xd - Xd.mean(0)
print('数据来源:', src, '| 形状', Xd.shape)
print('✅ 真实数据就绪')

**🧪 胶囊练习**：实现 `linear_ae_recon_error(X, d)`：在已中心化的 `X` 上，用 PCA 前 `d` 个主成分作为最优线性 AE，返回其**重建误差**（每样本平方和的均值）。它应当等于 PCA 丢弃的尾部特征值之和。

In [ ]:
def linear_ae_recon_error(X, d):
    # TODO: U = 前 d 个主成分; xhat = X@U@U.T; 返回 mean sum (xhat-X)^2
    raise NotImplementedError

In [ ]:
# 自测
U_d, evals_d = pca_subspace(Xd, 10)
err = linear_ae_recon_error(Xd, 10)
tail = evals_d[10:].sum()
print('digits 上 d=10: 线性 AE 重建误差 = %.4f, PCA 尾部和 = %.4f' % (err, tail))
assert abs(err - tail) < 1e-6, '真实数据上线性 AE 重建误差仍 = 丢弃特征值之和'
# 维度越多重建越好
assert linear_ae_recon_error(Xd, 20) < linear_ae_recon_error(Xd, 5)
print('✅ 胶囊练习通过：真实 digits 上「线性 AE = PCA」依然成立')

In [ ]:
# 📖 胶囊参考答案
def linear_ae_recon_error(X, d):
    U, _ = pca_subspace(X, d)
    xhat = (X @ U) @ U.T
    return np.mean(np.sum((xhat - X)**2, axis=1))

### 小结
- 自编码器 = 编码器 + 瓶颈 + 解码器，训练目标是重建损失；瓶颈逼它抓重点。
- 重建损失有概率含义：MSE ⇔ 高斯似然，BCE ⇔ 伯努利似然（VAE 重建项直接复用）。
- **线性 AE = PCA 子空间**，重建误差 = 丢弃的尾部特征值之和；非线性 AE 是 PCA 的非线性推广。
- 去噪 AE 加噪训练、重建干净数据，学到「往流形上拉」的方向（score 雏形，扩散的前身）。
- **但 AE 还采不了样**：隐空间无已知分布。

下一站：**模块 02 · VAE** —— 给隐空间加上概率结构与先验，第一次让模型能【采样】。